In [4]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Copy BrainTumorDataset class here (same as Notebook 3 Cell 2) ──
# (paste the full class definition)

class BrainTumorDataset(Dataset):
    def __init__(self, img_paths, mask_paths, augment=False, img_size=256):
        self.img_paths  = img_paths
        self.mask_paths = mask_paths
        self.img_size   = img_size
        self.augment    = augment

        self.aug_transforms = A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.3),
            A.RandomRotate90(p=0.5),
            A.ShiftScaleRotate(
                shift_limit=0.1, scale_limit=0.1,
                rotate_limit=15, p=0.4,
                border_mode=0
            ),
            A.ElasticTransform(alpha=120, sigma=120*0.05, p=0.2),
            A.RandomBrightnessContrast(
                brightness_limit=0.2,
                contrast_limit=0.2, p=0.4
            ),
            A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
            ToTensorV2()
        ])

        self.basic_transforms = A.Compose([
            A.Resize(img_size, img_size),
            ToTensorV2()
        ])

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        # Load image as grayscale
        img  = np.array(
            Image.open(self.img_paths[idx]).convert('L'),
            dtype=np.float32
        )
        mask = np.array(
            Image.open(self.mask_paths[idx]).convert('L'),
            dtype=np.float32
        )

        # Normalize image to [0, 1]
        img = img / 255.0

        # Convert mask to binary class labels
        # 0 = background, 1 = tumor
        mask_bin = np.zeros_like(mask, dtype=np.int64)
        mask_bin[mask > 127] = 1

        if self.augment:
            aug = self.aug_transforms(
                image=img,
                mask=mask_bin.astype(np.float32)
            )
        else:
            aug = self.basic_transforms(
                image=img,
                mask=mask_bin.astype(np.float32)
            )

        image_tensor = aug['image'].float()    # shape: (1, H, W)
        mask_tensor  = aug['mask'].long()      # shape: (H, W)

        return image_tensor, mask_tensor

# Load test data
test_df = pd.read_csv(r"C:\Users\Ismail\Documents\medical-segmentation\data\splits\test.csv")
test_dataset = BrainTumorDataset(
    test_df['image'].tolist(),
    test_df['mask'].tolist(),
    augment=False
)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=0)

# Load best model
model = smp.Unet(
    encoder_name="mobilenet_v2",  # ← must match training
    encoder_weights=None,
    in_channels=1, classes=2, activation=None
)
model.load_state_dict(
    torch.load(
        r"C:\Users\Ismail\Documents\medical-segmentation\notebooks\outputs\checkpoints\best_model.pth",
        map_location=device
    )
)
model = model.to(device)
model.eval()
print("✅ Model loaded successfully!")

C:\Users\Ismail\anaconda3\Lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
C:\Users\Ismail\AppData\Local\Temp\ipykernel_9824\2092224919.py:42: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),


✅ Model loaded successfully!


In [5]:
def dice_score(pred, target, cls=1, smooth=1.0):
    p = (pred == cls).astype(float)
    t = (target == cls).astype(float)
    intersection = (p * t).sum()
    return (2.0 * intersection + smooth) / (p.sum() + t.sum() + smooth)

def iou_score(pred, target, cls=1, smooth=1.0):
    p = (pred == cls).astype(float)
    t = (target == cls).astype(float)
    intersection = (p * t).sum()
    union        = p.sum() + t.sum() - intersection
    return (intersection + smooth) / (union + smooth)

def pixel_accuracy(pred, target):
    return (pred == target).sum() / target.size

In [6]:
dice_scores = []
iou_scores  = []
acc_scores  = []

with torch.no_grad():
    for imgs, masks in tqdm(test_loader, desc="Evaluating"):
        imgs  = imgs.to(device)
        preds = torch.softmax(model(imgs), dim=1)
        pred_masks = preds.argmax(dim=1).cpu().numpy()
        true_masks = masks.numpy()

        for pred, true in zip(pred_masks, true_masks):
            dice_scores.append(dice_score(pred, true, cls=1))
            iou_scores.append(iou_score(pred,  true, cls=1))
            acc_scores.append(pixel_accuracy(pred, true))

print("─" * 40)
print(f"  Dice Score     : {np.mean(dice_scores):.4f} ± {np.std(dice_scores):.4f}")
print(f"  IoU Score      : {np.mean(iou_scores):.4f} ± {np.std(iou_scores):.4f}")
print(f"  Pixel Accuracy : {np.mean(acc_scores):.4f} ± {np.std(acc_scores):.4f}")
print("─" * 40)

Evaluating: 100%|████████████████████████████████████████████████████████████████████| 115/115 [01:57<00:00,  1.02s/it]

────────────────────────────────────────
  Dice Score     : 0.0029 ± 0.0085
  IoU Score      : 0.0024 ± 0.0047
  Pixel Accuracy : 0.9830 ± 0.0135
────────────────────────────────────────
